# Imports
Load the libraries used throughout the advanced analytics workflow.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRADING_DAYS = 252
ROLLING_WINDOW = 90
KEY_FUND_COUNT = 5
MIN_SIP_TRANSACTIONS = 6
SIP_RISK_GAP_DAYS = 35

RISK_APPETITE_MAP = {
    "Low": ["Low"],
    "Moderate": ["Moderate", "Moderately High"],
    "High": ["High", "Very High"],
}

# Data Loading
Read the processed mutual fund datasets and inspect the core NAV table.

In [ ]:
nav = pd.read_csv(PROCESSED_DIR / "nav_history.csv", parse_dates=["date"])
scheme_performance = pd.read_csv(PROCESSED_DIR / "scheme_performance.csv")
investor_transactions = pd.read_csv(
    PROCESSED_DIR / "investor_transactions.csv",
    parse_dates=["transaction_date"],
)
portfolio_holdings = pd.read_csv(
    PROCESSED_DIR / "portfolio_holdings.csv",
    parse_dates=["portfolio_date"],
)
fund_master = pd.read_csv(PROCESSED_DIR / "fund_master.csv", parse_dates=["launch_date"])

nav.head()

# Helper Functions
Modular helper functions keep the analysis reusable, testable, and easier to maintain.

In [ ]:
def compute_daily_returns(nav_df: pd.DataFrame) -> pd.DataFrame:
    nav_df = nav_df.sort_values(["amfi_code", "date"]).copy()
    nav_df["daily_return"] = nav_df.groupby("amfi_code")["nav"].pct_change()
    return nav_df


def compute_var_cvar(nav_returns_df: pd.DataFrame, perf_df: pd.DataFrame) -> pd.DataFrame:
    risk_df = (
        nav_returns_df.dropna(subset=["daily_return"])
        .groupby("amfi_code")["daily_return"]
        .agg(
            VaR_95=lambda s: s.quantile(0.05),
            CVaR_95=lambda s: s[s <= s.quantile(0.05)].mean(),
        )
        .reset_index()
    )
    return (
        perf_df[["amfi_code", "scheme_name"]]
        .drop_duplicates()
        .merge(risk_df, on="amfi_code", how="left")
        .sort_values(["VaR_95", "CVaR_95", "scheme_name"], ascending=[True, True, True])
        .reset_index(drop=True)
    )


def compute_rolling_sharpe(nav_returns_df: pd.DataFrame) -> pd.DataFrame:
    nav_returns_df = nav_returns_df.sort_values(["amfi_code", "date"]).copy()
    grouped = nav_returns_df.groupby("amfi_code")["daily_return"]
    rolling_mean = grouped.transform(lambda s: s.rolling(ROLLING_WINDOW).mean())
    rolling_std = grouped.transform(lambda s: s.rolling(ROLLING_WINDOW).std())
    nav_returns_df["rolling_sharpe"] = (rolling_mean / rolling_std) * np.sqrt(TRADING_DAYS)
    return nav_returns_df


def select_key_funds(rolling_df: pd.DataFrame, perf_df: pd.DataFrame) -> pd.DataFrame:
    key_codes = (
        perf_df.sort_values(["aum_crore", "sharpe_ratio"], ascending=[False, False])
        .head(KEY_FUND_COUNT)["amfi_code"]
        .tolist()
    )
    meta = perf_df[["amfi_code", "scheme_name"]].drop_duplicates()
    return (
        rolling_df.loc[rolling_df["amfi_code"].isin(key_codes)]
        .merge(meta, on="amfi_code", how="left")
        .sort_values(["scheme_name", "date"])
        .reset_index(drop=True)
    )


def compute_cohort_analysis(transactions_df: pd.DataFrame, perf_df: pd.DataFrame) -> pd.DataFrame:
    first_year = (
        transactions_df.groupby("investor_id")["transaction_date"]
        .min()
        .dt.year.rename("cohort_year")
    )
    cohort_df = transactions_df.merge(first_year, on="investor_id", how="left")
    invested_mask = cohort_df["transaction_type"].isin(["SIP", "Lumpsum"])

    avg_sip_amount = (
        cohort_df.loc[cohort_df["transaction_type"] == "SIP"]
        .groupby("cohort_year")["amount_inr"]
        .mean()
        .rename("avg_sip_amount")
    )
    total_invested = (
        cohort_df.loc[invested_mask]
        .groupby("cohort_year")["amount_inr"]
        .sum()
        .rename("total_invested")
    )
    investor_count = (
        first_year.value_counts().rename_axis("cohort_year").rename("investor_count")
    )
    top_fund = (
        cohort_df.loc[invested_mask]
        .groupby(["cohort_year", "amfi_code"])["amount_inr"]
        .sum()
        .reset_index()
        .sort_values(["cohort_year", "amount_inr", "amfi_code"], ascending=[True, False, True])
        .drop_duplicates("cohort_year")
        .merge(perf_df[["amfi_code", "scheme_name"]], on="amfi_code", how="left")
        .set_index("cohort_year")["scheme_name"]
        .rename("top_fund")
    )
    return (
        pd.concat([avg_sip_amount, total_invested, investor_count, top_fund], axis=1)
        .reset_index()
        .sort_values("cohort_year")
        .reset_index(drop=True)
    )


def compute_sip_continuity(transactions_df: pd.DataFrame) -> pd.DataFrame:
    sip_df = transactions_df.loc[transactions_df["transaction_type"] == "SIP"].copy()
    sip_df = sip_df.sort_values(["investor_id", "transaction_date"])
    sip_df["gap_days"] = sip_df.groupby("investor_id")["transaction_date"].diff().dt.days
    eligible = sip_df.groupby("investor_id").filter(lambda frame: len(frame) >= MIN_SIP_TRANSACTIONS)
    status_df = (
        eligible.groupby("investor_id")["gap_days"]
        .mean()
        .reset_index(name="avg_gap_days")
        .sort_values(["avg_gap_days", "investor_id"], ascending=[False, True])
        .reset_index(drop=True)
    )
    status_df["status"] = np.where(status_df["avg_gap_days"] > SIP_RISK_GAP_DAYS, "at_risk", "healthy")
    return status_df


def recommend_funds(risk_appetite: str, perf_df: pd.DataFrame) -> pd.DataFrame:
    risk_grades = RISK_APPETITE_MAP[risk_appetite]
    ranked = (
        perf_df.loc[perf_df["risk_grade"].isin(risk_grades)]
        .sort_values(["sharpe_ratio", "aum_crore"], ascending=[False, False])
        .head(3)
        .copy()
    )
    return ranked[["scheme_name", "risk_grade", "sharpe_ratio", "return_3yr_pct", "aum_crore"]]


def compute_hhi(holdings_df: pd.DataFrame, master_df: pd.DataFrame) -> pd.DataFrame:
    equity_funds = master_df.loc[
        master_df["category"].str.contains("equity", case=False, na=False),
        ["amfi_code", "scheme_name"],
    ].drop_duplicates()
    return (
        holdings_df.merge(equity_funds, on="amfi_code", how="inner")
        .assign(weight_decimal=lambda df: df["weight_pct"] / 100)
        .groupby("scheme_name")["weight_decimal"]
        .apply(lambda s: np.square(s).sum())
        .reset_index(name="HHI")
        .sort_values(["HHI", "scheme_name"], ascending=[False, True])
        .reset_index(drop=True)
    )

In [ ]:
nav_returns = compute_daily_returns(nav)
var_cvar = compute_var_cvar(nav_returns, scheme_performance)
rolling_sharpe = compute_rolling_sharpe(nav_returns)
key_funds = select_key_funds(rolling_sharpe, scheme_performance)
cohort_summary = compute_cohort_analysis(investor_transactions, scheme_performance)
sip_continuity = compute_sip_continuity(investor_transactions)
hhi_summary = compute_hhi(portfolio_holdings, fund_master)
moderate_recommendations = recommend_funds("Moderate", scheme_performance)

# VaR and CVaR Analysis
Historical VaR uses the 5th percentile of daily returns, while CVaR measures the average loss beyond that threshold.

In [ ]:
var_cvar.head(10)

# Rolling Sharpe Analysis
The 90-day rolling Sharpe ratio shows how risk-adjusted returns evolve over time for five key funds.

In [ ]:
plt.figure(figsize=(14, 8))
for scheme_name, fund_data in key_funds.groupby("scheme_name", sort=False):
    plt.plot(fund_data["date"], fund_data["rolling_sharpe"], linewidth=2, label=scheme_name)

plt.axhline(0, color="#666666", linestyle="--", linewidth=1)
plt.title("Rolling 90-Day Sharpe Ratio")
plt.xlabel("Date")
plt.ylabel("Sharpe Ratio")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

# Cohort Analysis
Investors are grouped by their first transaction year to compare periodic commitment and cumulative invested amounts.

In [ ]:
cohort_summary.head(10)

# SIP Continuity Analysis
Investors with at least six SIP transactions are flagged based on their average gap between installments.

In [ ]:
sip_continuity.head(10)

# Recommender System
Funds are mapped to risk appetite buckets and ranked by Sharpe ratio in descending order.

In [ ]:
recommend_funds("Moderate", scheme_performance)

# HHI Concentration
Portfolio concentration is measured with the Herfindahl-Hirschman Index using disclosed holding weights.

In [ ]:
hhi_summary.head(10)

# Advanced Insights
1. Highest tail risk sits with **SBI Small Cap Fund - Direct Plan - Growth**, where the 95% historical VaR is **-0.0269** and the conditional loss measure CVaR is **-0.0324**.
2. Cohort **2024** invests the most overall, contributing **INR 2,258,062,304** with **Axis Small Cap Fund - Regular - Growth** as the top preference.
3. SIP continuity is healthy for **2.2%** of investors who made at least 6 SIPs, leaving the remainder flagged as at risk.
4. The most diversified equity fund is **SBI Small Cap Fund - Regular Plan - Growth** with HHI **0.107**, while **Axis Bluechip Fund - Regular - Growth** is the most concentrated at **0.206**.
5. On risk-adjusted returns, **ICICI Pru Liquid Fund - Regular - Growth** leads the universe with a Sharpe ratio of **7.68**.